以下を行なう。
```
Gaussian distribution
        ↓
   Generative Model
        ↓
latent distribution
        ↓
      Decoder
        ↓
digit image
```



In [ ]:
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits
from torch.utils.data import DataLoader, TensorDataset

# -----------------------
# setup
# -----------------------
device = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(0)
np.random.seed(0)

# -----------------------
# data: sklearn digits
# -----------------------
digits = load_digits()
x = digits.data.astype("float32") / 16.0      # [N, 64]
y = digits.target

x_tensor = torch.tensor(x)
dataset = TensorDataset(x_tensor)
loader = DataLoader(dataset, batch_size=128, shuffle=True)

# -----------------------
# Autoencoder
# -----------------------
latent_dim = 2

class AutoEncoder(nn.Module):
    def __init__(self, latent_dim=2):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(64, 128),
            nn.GELU(),
            nn.Linear(128, 64),
            nn.GELU(),
            nn.Linear(64, latent_dim),
        )
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 64),
            nn.GELU(),
            nn.Linear(64, 128),
            nn.GELU(),
            nn.Linear(128, 64),
            nn.Sigmoid(),
        )

    def encode(self, x):
        return self.encoder(x)

    def decode(self, z):
        return self.decoder(z)

    def forward(self, x):
        z = self.encode(x)
        x_rec = self.decode(z)
        return x_rec, z

ae = AutoEncoder(latent_dim=latent_dim).to(device)
opt_ae = torch.optim.Adam(ae.parameters(), lr=1e-3)

# train AE
for epoch in range(100):
    total = 0
    for (xb,) in loader:
        xb = xb.to(device)
        x_rec, z = ae(xb)
        loss = ((x_rec - xb) ** 2).mean()

        opt_ae.zero_grad()
        loss.backward()
        opt_ae.step()

        total += loss.item()

    if epoch % 20 == 0:
        print("AE epoch", epoch, "loss", total / len(loader))

# latent dataset
ae.eval()
with torch.no_grad():
    z_data = ae.encode(x_tensor.to(device)).cpu()

z_mean = z_data.mean(dim=0, keepdim=True)
z_std = z_data.std(dim=0, keepdim=True)
z_data_norm = (z_data - z_mean) / z_std

latent_dataset = TensorDataset(z_data_norm)
latent_loader = DataLoader(latent_dataset, batch_size=256, shuffle=True)

# -----------------------
# Flow Matching model in latent space
# -----------------------
class FlowModel(nn.Module):
    def __init__(self, latent_dim=2, hidden=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(latent_dim + 1, hidden),
            nn.GELU(),
            nn.Linear(hidden, hidden),
            nn.GELU(),
            nn.Linear(hidden, latent_dim),
        )

    def forward(self, z_t, t):
        return self.net(torch.cat([z_t, t], dim=1))

flow = FlowModel(latent_dim=latent_dim).to(device)
opt_flow = torch.optim.Adam(flow.parameters(), lr=1e-3)

# train Flow Matching
for epoch in range(300):
    total = 0

    for (z1,) in latent_loader:
        z1 = z1.to(device)
        z0 = torch.randn_like(z1)
        t = torch.rand(z1.shape[0], 1, device=device)

        z_t = (1 - t) * z0 + t * z1
        v_target = z1 - z0

        v_pred = flow(z_t, t)
        loss = ((v_pred - v_target) ** 2).mean()

        opt_flow.zero_grad()
        loss.backward()
        opt_flow.step()

        total += loss.item()

    if epoch % 50 == 0:
        print("FM epoch", epoch, "loss", total / len(latent_loader))

# -----------------------
# sampling latent paths
# -----------------------
@torch.no_grad()
def sample_latent_paths(model, n_samples=5000, n_steps=100):
    model.eval()

    z = torch.randn(n_samples, latent_dim, device=device)
    paths = torch.zeros(n_samples, n_steps + 1, latent_dim, device=device)
    paths[:, 0] = z

    ts = torch.linspace(0, 1, n_steps + 1, device=device)

    for i in range(n_steps):
        t = ts[i].expand(n_samples, 1)
        dt = ts[i + 1] - ts[i]
        z = z + model(z, t) * dt
        paths[:, i + 1] = z

    return paths.cpu()

In [ ]:
paths = sample_latent_paths(flow, n_samples=10000, n_steps=150)

# -----------------------
# Path Density: z1 direction only
# -----------------------
z_index = 0
n_steps = paths.shape[1] - 1

z_min = -4
z_max = 4
n_bins = 300

density = np.zeros((n_steps + 1, n_bins))
edges = np.linspace(z_min, z_max, n_bins + 1)

for i in range(n_steps + 1):
    density[i] = np.histogram(paths[:, i, z_index].numpy(), bins=edges)[0]

plt.figure(figsize=(9, 5))

plt.imshow(
    density.T,
    origin="lower",
    aspect="auto",
    extent=[0, 1, z_min, z_max],
    cmap="viridis"
)

plt.colorbar(label="count")
plt.xlabel("t")
plt.ylabel("latent z1")
plt.title("Path Density in Autoencoder Latent Space")

# -----------------------
# velocity arrows on z1 axis
# -----------------------
t_grid = np.linspace(0, 1, 15)
z_grid = np.linspace(z_min, z_max, 25)

T, Z1 = np.meshgrid(t_grid, z_grid)

# z2 = 0 slice
Z2 = np.zeros_like(Z1)

z_tensor = torch.tensor(
    np.stack([Z1.reshape(-1), Z2.reshape(-1)], axis=1),
    dtype=torch.float32,
    device=device
)

t_tensor = torch.tensor(
    T.reshape(-1, 1),
    dtype=torch.float32,
    device=device
)

with torch.no_grad():
    V_all = flow(z_tensor, t_tensor).cpu().numpy()

V = V_all[:, z_index].reshape(Z1.shape)

U = np.ones_like(V)

norm = np.sqrt(U**2 + V**2)
U_plot = U / norm
V_plot = V / norm

plt.quiver(
    T,
    Z1,
    U_plot,
    V_plot,
    color="white",
    angles="xy",
    scale_units="xy",
    scale=25,
    width=0.003,
    alpha=0.8
)

plt.tight_layout()
plt.show()

# -----------------------
# generate digit images
# -----------------------
@torch.no_grad()
def generate_images(model, ae, n_samples=64, n_steps=100):
    model.eval()
    ae.eval()

    z = torch.randn(n_samples, latent_dim, device=device)
    ts = torch.linspace(0, 1, n_steps + 1, device=device)

    for i in range(n_steps):
        t = ts[i].expand(n_samples, 1)
        dt = ts[i + 1] - ts[i]
        z = z + model(z, t) * dt

    # unnormalize latent
    z = z.cpu() * z_std + z_mean
    z = z.to(device)

    x_gen = ae.decode(z)
    return x_gen.cpu().numpy()

imgs = generate_images(flow, ae, n_samples=64, n_steps=150)

fig, axes = plt.subplots(8, 8, figsize=(6, 6))

for i, ax in enumerate(axes.ravel()):
    ax.imshow(imgs[i].reshape(8, 8), cmap="gray")
    ax.axis("off")

plt.suptitle("Generated digits by Autoencoder Latent Flow Matching")
plt.tight_layout()
plt.show()

In [ ]:
# -----------------------
# 2D latent density + velocity arrows
# -----------------------

# t = 0, 0.1, ..., 1.0 に対応する index
show_indices = [int(n_steps * k / 10) for k in range(11)]
show_times = [k / 10 for k in range(11)]

z_min = -4
z_max = 4

fig, axes = plt.subplots(
    3, 4,
    figsize=(12, 9),
    sharex=True,
    sharey=True
)

axes = axes.ravel()

# velocity field grid
grid_size = 15
z1_grid = np.linspace(z_min, z_max, grid_size)
z2_grid = np.linspace(z_min, z_max, grid_size)
Z1, Z2 = np.meshgrid(z1_grid, z2_grid)

z_grid_tensor = torch.tensor(
    np.stack([Z1.reshape(-1), Z2.reshape(-1)], axis=1),
    dtype=torch.float32,
    device=device
)

for ax, idx, tval in zip(axes, show_indices, show_times):
    z = paths[:, idx, :].numpy()

    ax.hist2d(
        z[:, 0],
        z[:, 1],
        bins=80,
        range=[[z_min, z_max], [z_min, z_max]],
        cmap="viridis"
    )

    t_tensor = torch.full(
        (z_grid_tensor.shape[0], 1),
        tval,
        dtype=torch.float32,
        device=device
    )

    with torch.no_grad():
        V = flow(z_grid_tensor, t_tensor).cpu().numpy()

    U = V[:, 0].reshape(Z1.shape)
    W = V[:, 1].reshape(Z2.shape)

    ax.quiver(
        Z1,
        Z2,
        U,
        W,
        color="white",
        angles="xy",
        scale_units="xy",
        scale=35,
        width=0.004,
        alpha=0.8
    )

    ax.set_title(f"t = {tval:.1f}")
    ax.set_xlim(z_min, z_max)
    ax.set_ylim(z_min, z_max)
    ax.set_aspect("equal")
    ax.set_xlabel("z1")
    ax.set_ylabel("z2")

for j in range(len(show_indices), len(axes)):
    axes[j].axis("off")

plt.suptitle("2D Latent Path Density with Flow Velocity")
plt.tight_layout()
plt.show()

In [ ]:
# -----------------------
# Decode 10x10 grid on t=1 latent space
# -----------------------

@torch.no_grad()
def plot_decoder_grid(ae, z_mean, z_std, z_min=-4, z_max=4, grid_size=10):
    ae.eval()

    z1_values = np.linspace(z_min, z_max, grid_size)
    z2_values = np.linspace(z_min, z_max, grid_size)

    fig, axes = plt.subplots(
        grid_size,
        grid_size,
        figsize=(10, 10),
        sharex=True,
        sharey=True
    )

    for i, z2 in enumerate(z2_values[::-1]):
        for j, z1 in enumerate(z1_values):
            # normalized latent coordinate
            z_norm = torch.tensor(
                [[z1, z2]],
                dtype=torch.float32,
                device=device
            )

            # unnormalize latent because decoder was trained on original AE latent
            z = z_norm.cpu() * z_std + z_mean
            z = z.to(device)

            x_dec = ae.decode(z)
            img = x_dec.cpu().numpy().reshape(8, 8)

            ax = axes[i, j]
            ax.imshow(img, cmap="gray", vmin=0, vmax=1)
            ax.set_title(f"{z1:.1f},{z2:.1f}", fontsize=7)
            ax.axis("off")

    plt.suptitle("Decoder output on 10x10 latent grid at t=1")
    plt.tight_layout()
    plt.show()


plot_decoder_grid(
    ae=ae,
    z_mean=z_mean,
    z_std=z_std,
    z_min=-2,
    z_max=2,
    grid_size=10
)